Authors: Matt Kuehr, Matthew Sawyer

Emails: mck0063@auburn.edu, mss0096@auburn.edu

# Out-of-Distribution (OOD) Evaluation

This notebook evaluates the pre-trained LSTM model (trained on the Sarcasm News Headline dataset) on a curated set of out-of-distribution examples from `ood/train1.json` and `ood/train2.json`.

In [2]:
import os
import sys
sys.path.append('..')
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from scripts.models import SentimentRNN
from scripts.preprocess import collate_fn

c:\Users\Matt\Desktop\old_repo\COMP-6630-Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
TOKENIZER_PATH = "../tokenizer/tokenizer_word.json"
OOD_FILES = ["../ood_data/train1.json", "../ood_data/train2.json"]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

EMBED_DIM = 64
HIDDEN_DIM = 128
OUTPUT_DIM = 1
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5
MODEL_TYPE = "lstm"

In [4]:
# Generate paths for checkpoints (all checkpoints + best model checkpoint)
MODEL_PATHS = [f"../checkpoints/model_epoch_{i}.pt" for i in range(1, 26)] + ["../checkpoints/best_model.pt"]

In [5]:
class OODDataset(Dataset):
    """
    A custom PyTorch Dataset for Out-of-Distribution (OOD) evaluation.

    Attributes:
        data (list): A list of dictionaries containing headlines and labels.
        tokenizer (Tokenizer): The tokenizer used to encode headlines.
    """
    def __init__(self, json_files, tokenizer):
        """
        Initializes the OODDataset by loading data from JSON files.

        Args:
            json_files (list): A list of paths to JSON files containing OOD data.
            tokenizer (Tokenizer): The tokenizer for encoding headlines.
        """
        self.data = []
        self.tokenizer = tokenizer
        
        label_map = {"real": 0, "fake": 1}
        
        for file_path in json_files:
            if not os.path.exists(file_path):
                continue
            with open(file_path, 'r', encoding='utf-8') as f:
                batch_data = json.load(f)
                for item in batch_data:
                    label = item["label"]
                    if isinstance(label, str):
                        label = label_map.get(label.lower(), 0)
                    
                    self.data.append({
                        "headline": item["headline"],
                        "label": label
                    })
                    
    def __len__(self):
        """
        Returns the total number of samples in the OOD dataset.

        Returns:
            int: The number of samples.
        """
        return len(self.data)

    def __getitem__(self, idx):
        """
        Retrieves a single sample from the OOD dataset.

        Args:
            idx (int): The index of the sample to retrieve.

        Returns:
            dict: A dictionary containing 'input_ids' and 'label'.
        """
        item = self.data[idx]
        encoding = self.tokenizer.encode(item["headline"])
        
        return {
            "input_ids": torch.tensor(encoding.ids, dtype=torch.long),
            "label": torch.tensor(item["label"], dtype=torch.float)
        }

In [6]:
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
vocab_size = tokenizer.get_vocab_size()
print(f"Loaded tokenizer with vocab size: {vocab_size}")

ood_dataset = OODDataset(OOD_FILES, tokenizer)
ood_loader = DataLoader(ood_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
print(f"Loaded {len(ood_dataset)} OOD examples.")

Loaded tokenizer with vocab size: 26486
Loaded 100 OOD examples.


In [7]:
def evaluate_ood(model, iterator, device):
    """
    Evaluates the model on the Out-of-Distribution (OOD) iterator.

    Args:
        model (nn.Module): The model to be evaluated.
        iterator (DataLoader): The DataLoader for OOD data.
        device (torch.device): The device to run evaluation on.

    Returns:
        tuple: A tuple containing (average_epoch_loss, average_epoch_accuracy).
    """
    epoch_loss = 0
    epoch_acc = 0
    criterion = nn.BCELoss()
    
    model.eval()
    with torch.no_grad():
        for batch in iterator:
            text = batch["input_ids"].to(device)
            labels = batch["label"].unsqueeze(1).to(device)
            lengths = batch["lengths"]
            
            predictions = model(text, lengths)
            loss = criterion(predictions, labels)
            
            rounded_preds = torch.round(predictions)
            correct = (rounded_preds == labels).float()
            acc = correct.sum() / len(correct)
            
            epoch_loss += loss.item()
            epoch_acc += acc.item()
            
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

print(f"{'Checkpoint':<30} | {'Loss':<6} | {'Accuracy':<8}")
print("-" * 50)

for model_path in MODEL_PATHS:
    if not os.path.exists(model_path):
        continue
        
    model = SentimentRNN(vocab_size, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM, 
                         N_LAYERS, BIDIRECTIONAL, DROPOUT, model_type=MODEL_TYPE)
    checkpoint = torch.load(model_path, map_location=DEVICE)
    
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    else:
        state_dict = checkpoint
        
    model.load_state_dict(state_dict)
    model = model.to(DEVICE)
    
    loss, acc = evaluate_ood(model, ood_loader, DEVICE)
    print(f"{model_path:<30} | {loss:.3f}  | {acc*100:6.2f}%")

Checkpoint                     | Loss   | Accuracy
--------------------------------------------------
../checkpoints/model_epoch_1.pt | 0.638  |  65.62%
../checkpoints/model_epoch_2.pt | 0.759  |  67.19%
../checkpoints/model_epoch_3.pt | 0.756  |  65.62%
../checkpoints/model_epoch_4.pt | 0.576  |  70.31%
../checkpoints/model_epoch_5.pt | 0.693  |  71.88%
../checkpoints/model_epoch_6.pt | 0.914  |  72.66%
../checkpoints/model_epoch_7.pt | 0.735  |  75.00%
../checkpoints/model_epoch_8.pt | 0.787  |  72.66%
../checkpoints/model_epoch_9.pt | 0.873  |  71.09%
../checkpoints/model_epoch_10.pt | 0.821  |  74.22%
../checkpoints/model_epoch_11.pt | 0.977  |  71.09%
../checkpoints/model_epoch_12.pt | 0.791  |  73.44%
../checkpoints/model_epoch_13.pt | 1.009  |  70.31%
../checkpoints/model_epoch_14.pt | 1.112  |  71.09%
../checkpoints/model_epoch_15.pt | 1.005  |  73.44%
../checkpoints/model_epoch_16.pt | 1.316  |  71.88%
../checkpoints/model_epoch_17.pt | 0.978  |  71.09%
../checkpoints/model_ep

In [15]:
best_model_path = "../checkpoints/best_model.pt"
if os.path.exists(best_model_path):
    model = SentimentRNN(vocab_size, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM, 
                         N_LAYERS, BIDIRECTIONAL, DROPOUT, model_type=MODEL_TYPE)
    model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
    model = model.to(DEVICE)
    
    print("\nDetailed Results for Best Model:\n")
    for ood_file in OOD_FILES:
        if not os.path.exists(ood_file):
            continue
        ds = OODDataset([ood_file], tokenizer)
        loader = DataLoader(ds, batch_size=32, shuffle=False, collate_fn=collate_fn)
        file_loss, file_acc = evaluate_ood(model, loader, DEVICE)
        print(f"Loss on {ood_file}: {file_loss:.4f}")
        print(f"Accuracy on {ood_file}: {file_acc*100:.2f}%")
        print()


Detailed Results for Best Model:

Loss on ../ood_data/train1.json: 2.1299
Accuracy on ../ood_data/train1.json: 55.73%

Loss on ../ood_data/train2.json: 0.6332
Accuracy on ../ood_data/train2.json: 82.81%

